# Eigenvalues and Eigenvectors

Companion notebook for the [Eigenvalues and Eigenvectors](https://ml-viz.vercel.app/courses/linear-algebra/03-eigenvalues-and-eigenvectors) lesson.

We'll compute eigendecompositions, visualize eigenvectors geometrically, and implement PCA from scratch.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Computing eigenvalues and eigenvectors

In [ ]:
A = np.array([[4., 1.], [2., 3.]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print('Matrix A:')
print(A)
print('\nEigenvalues:', eigenvalues)
print('Eigenvectors (columns):')
print(eigenvectors)

# Verify: A @ v == λ * v
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    av  = A @ v
    lv  = lam * v
    print(f'\nλ={lam:.1f}: Av={av.round(4)}, λv={lv.round(4)}, equal={np.allclose(av,lv)}')

## Deriving eigenvalues by hand

`np.linalg.eig` is a black box. Here we reproduce it from the definition. Setting $\det(\mathbf{A} - \lambda\mathbf{I}) = 0$ for a $2\times2$ matrix gives the characteristic polynomial

$$\lambda^2 - \operatorname{tr}(\mathbf{A})\,\lambda + \det(\mathbf{A}) = 0,$$

whose roots are the eigenvalues. Each eigenvector is then a null-space direction of $(\mathbf{A} - \lambda\mathbf{I})$. As a sanity check, the eigenvalues must satisfy $\sum_i \lambda_i = \operatorname{tr}(\mathbf{A})$ and $\prod_i \lambda_i = \det(\mathbf{A})$.

In [ ]:
# Derive eigenvalues BY HAND from the characteristic equation det(A - λI) = 0.
# For 2x2 this is  λ² - tr(A)·λ + det(A) = 0.
tr  = np.trace(A)
det = np.linalg.det(A)
print(f'characteristic poly:  λ² - {tr:.0f}λ + {det:.0f} = 0')

disc = tr**2 - 4 * det
lam1 = (tr + np.sqrt(disc)) / 2
lam2 = (tr - np.sqrt(disc)) / 2
print(f'roots (eigenvalues):  λ1={lam1:.0f}, λ2={lam2:.0f}')

# Eigenvector = null space direction of (A - λI). For a 2x2 singular matrix
# [[a,b],[c,d]] a null vector is [b, -a] (or [-d, c]); pick whichever is non-zero.
def eigvec(A, lam):
    M = A - lam * np.eye(2)
    v = np.array([M[0, 1], -M[0, 0]])
    if np.allclose(v, 0):
        v = np.array([-M[1, 1], M[1, 0]])
    return v / np.linalg.norm(v)

for lam in (lam1, lam2):
    v = eigvec(A, lam)
    print(f'λ={lam:.0f}: eigenvector {v.round(3)}  check A·v - λ·v = {(A @ v - lam * v).round(6)}')

# Eigenvalues tie back to trace and determinant:
print(f'\nsum of eigenvalues  = {lam1 + lam2:.0f}  = trace(A) = {tr:.0f}')
print(f'product of eigenvalues = {lam1 * lam2:.0f}  = det(A)  = {det:.0f}')

## Eigenvectors define special directions

For a random vector the transformation changes both direction and magnitude.
For an eigenvector, only the magnitude changes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

colors = ['#6366f1', '#2dd4bf', '#f97316']

# Plot eigenvectors and their transforms
for i, color in zip(range(2), colors[:2]):
    v   = eigenvectors[:, i]
    Av  = A @ v
    lam = eigenvalues[i]

    ax.annotate('', xy=v, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.annotate('', xy=Av, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color=color, lw=2, linestyle='--', alpha=0.6))
    ax.text(v[0]*1.05, v[1]*1.05 + 0.1, f'v₁  (λ={lam:.0f})', color=color, fontsize=11)

# Random vector
r  = np.array([1.0, 0.3])
Ar = A @ r
ax.annotate('', xy=r, xytext=[0,0], arrowprops=dict(arrowstyle='->', color=colors[2], lw=2))
ax.annotate('', xy=Ar, xytext=[0,0],
            arrowprops=dict(arrowstyle='->', color=colors[2], lw=2, linestyle='--', alpha=0.6))
ax.text(r[0]*1.1, r[1]+0.1, 'random →', color=colors[2], fontsize=10)
ax.text(Ar[0]*1.05, Ar[1]+0.1, '→ A·random', color=colors[2], alpha=0.7, fontsize=10)

ax.set_xlim(-0.3, 2); ax.set_ylim(-0.3, 2.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
ax.axhline(0, color='#30344a'); ax.axvline(0, color='#30344a')
ax.set_title('Eigenvectors stay in the same direction after transformation', pad=12)
plt.tight_layout(); plt.show()

## PCA from scratch using eigendecomposition

In [ ]:
rng = np.random.default_rng(42)

# Correlated 2D data
cov_true = np.array([[3., 2.], [2., 2.]])
X = rng.multivariate_normal([0, 0], cov_true, size=200)

# PCA via eigendecomposition
C = (X.T @ X) / len(X)                    # sample covariance
eigenvalues, V = np.linalg.eigh(C)          # eigh for symmetric
idx = np.argsort(eigenvalues)[::-1]         # sort descending
eigenvalues, V = eigenvalues[idx], V[:, idx]

explained = eigenvalues / eigenvalues.sum()
print('Eigenvalues:', eigenvalues.round(4))
print('Variance explained:', explained.round(4))
print('PC1 direction:', V[:, 0].round(4))

# Project to 1D
X_1d = X @ V[:, 0]       # principal component scores

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], alpha=0.4, s=15, color='#6366f1')
scale = 2
for i, color in enumerate(['#f97316', '#2dd4bf']):
    v = V[:, i] * np.sqrt(eigenvalues[i]) * scale
    ax.annotate('', xy=v, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(v[0]+0.1, v[1]+0.1, f'PC{i+1} ({explained[i]*100:.0f}%)', color=color, fontsize=11)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
ax.set_title('Original data + Principal Components')

ax2 = axes[1]
ax2.hist(X_1d, bins=30, color='#6366f1', alpha=0.8, edgecolor='#0f1117')
ax2.set_title(f'Projected onto PC1 ({explained[0]*100:.0f}% variance)')
ax2.set_xlabel('PC1 score'); ax2.set_ylabel('Count')

plt.tight_layout(); plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Eigenvalues from trace and determinant

For a 2×2 matrix, the characteristic polynomial collapses to a quadratic in two summaries you already know:

$$\lambda^2 - \text{tr}(A)\,\lambda + \det(A) = 0
\quad\Rightarrow\quad
\lambda = \frac{\text{tr}(A) \pm \sqrt{\text{tr}(A)^2 - 4\det(A)}}{2}$$

Implement it (assume real eigenvalues) and match `np.linalg.eigvals`.

In [ ]:
def eigvals_2x2(A):
    """Both eigenvalues of a 2x2 matrix (assumed real), returned (larger, smaller)."""
    A = np.asarray(A, dtype=float)

    # TODO(you): trace = sum of the diagonal
    tr = ...

    # TODO(you): determinant ad - bc
    det = ...

    # TODO(you): square root of the discriminant tr^2 - 4*det
    disc = ...

    return ((tr + disc) / 2, (tr - disc) / 2)

In [ ]:
# Checks — run me
big, small = eigvals_2x2([[4, 1], [2, 3]])
assert abs(big - 5) < 1e-12 and abs(small - 2) < 1e-12, "expected eigenvalues 5 and 2"

big, small = eigvals_2x2([[3, 0], [0, -1]])
assert abs(big - 3) < 1e-12 and abs(small + 1) < 1e-12, "diagonal matrix: eigenvalues sit on the diagonal"

A = np.array([[2.0, 1.0], [1.0, 2.0]])
assert np.allclose(sorted(eigvals_2x2(A)), sorted(np.linalg.eigvals(A).real)), "must match np.linalg.eigvals"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def eigvals_2x2(A):
    A = np.asarray(A, dtype=float)
    tr = A[0, 0] + A[1, 1]
    det = A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
    disc = np.sqrt(tr ** 2 - 4 * det)
    return ((tr + disc) / 2, (tr - disc) / 2)
```

</details>

### Exercise 2 — Power iteration

Repeatedly applying $A$ to *any* vector rotates it toward the **dominant eigenvector** — the component along the largest eigenvalue grows fastest, so after renormalizing each step, it's all that survives. The eigenvalue then falls out of the Rayleigh quotient $\lambda = \mathbf{v}^\top A \mathbf{v}$ (for unit $\mathbf{v}$). This is the idea behind PageRank and many sparse eigensolvers.

In [ ]:
def power_iteration(A, iters=100):
    """Return (dominant eigenvalue, unit eigenvector) by repeated multiplication."""
    A = np.asarray(A, dtype=float)
    v = np.ones(A.shape[0])

    for _ in range(iters):
        # TODO(you): multiply v by A
        v = ...
        # TODO(you): renormalize v to unit length
        v = ...

    # TODO(you): Rayleigh quotient v·(A v)  (v already has unit length)
    lam = ...

    return lam, v

In [ ]:
# Checks — run me
A = np.array([[3.0, 2.0], [2.0, 2.0]])
lam, v = power_iteration(A)

true_lams, true_V = np.linalg.eigh(A)
assert abs(lam - true_lams[-1]) < 1e-8, "should find the LARGEST eigenvalue"
assert np.linalg.norm(A @ v - lam * v) < 1e-8, "v must satisfy A v = λ v"
assert abs(abs(v @ true_V[:, -1]) - 1) < 1e-8, "v must align with the dominant eigenvector"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def power_iteration(A, iters=100):
    A = np.asarray(A, dtype=float)
    v = np.ones(A.shape[0])
    for _ in range(iters):
        v = A @ v
        v = v / np.linalg.norm(v)
    lam = v @ A @ v
    return lam, v
```

</details>